<a href="https://colab.research.google.com/github/sami01979/ai_text_detection/blob/main/ai_text_detection_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

MODEL = "roberta-base"
MAX_LEN = 384

train = pd.read_csv("train.csv"); val = pd.read_csv("val.csv"); test = pd.read_csv("test.csv")
tok = AutoTokenizer.from_pretrained(MODEL)

def prep(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    return ds.remove_columns(["text"])

tr, va, te = prep(train), prep(val), prep(test)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="out_big",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=300,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
trainer.train()
trainer.save_model("final_model_v1")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.146790,0.246243,0.933072,0.896266,0.979497,0.936035
2,0.085107,0.427372,0.919204,0.867230,0.989950,0.924535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
from scipy.special import softmax
from sklearn.metrics import roc_auc_score, classification_report

val_out = trainer.predict(va)
val_p = softmax(val_out.predictions, axis=-1)[:, 1]
best_t, best_acc = 0.5, 0
for t in np.arange(0.05, 0.96, 0.01):
    acc = accuracy_score(val_out.label_ids, (val_p >= t).astype(int))
    if acc > best_acc: best_t, best_acc = t, acc
print("best threshold (chosen on VALIDATION):", round(best_t, 2), "| val acc:", round(best_acc, 4))

test_out = trainer.predict(te)
test_p = softmax(test_out.predictions, axis=-1)[:, 1]
y = test_out.label_ids
print("TEST acc @0.5   :", round(accuracy_score(y, (test_p >= 0.5).astype(int)), 4))
print("TEST acc @tuned :", round(accuracy_score(y, (test_p >= best_t).astype(int)), 4))
print("ROC-AUC:", round(roc_auc_score(y, test_p), 4))
final_pred = (test_p >= best_t).astype(int)
print(confusion_matrix(y, final_pred))
print(classification_report(y, final_pred, target_names=["Human", "AI"], digits=4))

best threshold (chosen on VALIDATION): 0.95 | val acc: 0.9455


TEST acc @0.5   : 0.9385
TEST acc @tuned : 0.9485
ROC-AUC: 0.9906
[[4646  330]
 [ 183 4793]]
              precision    recall  f1-score   support

       Human     0.9621    0.9337    0.9477      4976
          AI     0.9356    0.9632    0.9492      4976

    accuracy                         0.9485      9952
   macro avg     0.9488    0.9485    0.9484      9952
weighted avg     0.9488    0.9485    0.9484      9952



In [5]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r final_model_v1 /content/drive/MyDrive/

Mounted at /content/drive


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

path = "final_model_v1"   # or "/content/drive/MyDrive/final_model_v1"
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def detect(text):
    inputs = tok(text, truncation=True, max_length=384, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    ai_prob = probs[1].item()
    label = "AI-generated" if ai_prob >= 0.5 else "Human-written"
    print(f"{label}  (AI probability: {ai_prob:.1%})")

detect("hey , what do you think is it ai written or human written , if i want to trick you with ai and human mixed will you able to figure it out?")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Human-written  (AI probability: 0.1%)


In [7]:
text1 = """
Yesterday I went to the market with my friend. We were supposed
to buy a few things, but somehow we ended up walking around for
almost two hours. I forgot to buy the one thing I actually went
there for, which was a phone charger. I only remembered when I
got home. Honestly, this happens to me way too often.
"""

detect(text1)

Human-written  (AI probability: 3.4%)


In [8]:
text2 = """
Artificial intelligence has become an important component of
modern education. Machine learning algorithms can analyze
student performance, identify learning difficulties, and provide
personalized educational recommendations. However, the
implementation of these technologies also raises concerns
regarding data privacy, algorithmic bias, and the reliability
of automated decision-making systems.
"""

detect(text2)

AI-generated  (AI probability: 86.7%)


In [9]:
text3 = """
A machine learning classifier learns patterns from a training
dataset and uses those patterns to make predictions on unseen
data. The performance of a classifier depends on the quality
of the training samples, the selected features, and the
evaluation methodology. Data leakage can produce misleading
results by allowing information from the test dataset to
influence the training process.
"""

detect(text3)

Human-written  (AI probability: 5.3%)


In [10]:
texts = [
    text1,
    text2,
    text3
]

for i, text in enumerate(texts, 1):
    print(f"\nTest {i}")
    detect(text)


Test 1
Human-written  (AI probability: 3.4%)

Test 2
AI-generated  (AI probability: 86.7%)

Test 3
Human-written  (AI probability: 5.3%)


In [11]:
texts = {
    "Casual": """
    I missed my bus this morning, so I had to wait another
    twenty minutes. I was already late for class, and to make
    things worse, it started raining just as I got there.
    """,

    "Academic": """
    The integration of artificial intelligence into educational
    systems has created opportunities for personalized learning.
    Nevertheless, concerns regarding privacy, fairness, and
    transparency remain significant challenges.
    """,

    "Technical": """
    A binary classification model maps an input feature vector
    to one of two predicted classes. Its performance can be
    evaluated using precision, recall, F1-score, and ROC-AUC.
    """,

    "Personal reflection": """
    I used to think learning programming meant memorizing
    commands. After working on a few projects, I realized that
    understanding problems and debugging errors matter much more.
    """,

    "News style": """
    Researchers are investigating new approaches to automated
    text analysis as generative artificial intelligence becomes
    increasingly common in education and publishing.
    """,

    "Explanatory": """
    Photosynthesis is the process through which plants use
    sunlight to convert water and carbon dioxide into sugars.
    This process also releases oxygen into the atmosphere.
    """,

    "Opinion": """
    Technology can make education more accessible, but students
    should not become completely dependent on automated tools.
    Developing independent thinking remains essential.
    """,

    "Storytelling": """
    The library was almost empty when I arrived. I chose a seat
    near the window, opened my notebook, and realized I had
    forgotten to bring the book I needed for the assignment.
    """,

    "Business": """
    Small businesses can use data analysis to understand customer
    preferences, improve inventory planning, and make better
    decisions about pricing and product selection.
    """,

    "Conversational": """
    Honestly, I did not expect the project to take this long.
    Every time I fixed one problem, another appeared. At least
    I learned something new along the way.
    """
}

for style, text in texts.items():
    print(f"\n--- {style} ---")
    detect(text)


--- Casual ---
AI-generated  (AI probability: 93.7%)

--- Academic ---
AI-generated  (AI probability: 99.1%)

--- Technical ---
AI-generated  (AI probability: 94.3%)

--- Personal reflection ---
AI-generated  (AI probability: 96.5%)

--- News style ---
AI-generated  (AI probability: 97.2%)

--- Explanatory ---
AI-generated  (AI probability: 98.7%)

--- Opinion ---
AI-generated  (AI probability: 98.1%)

--- Storytelling ---
AI-generated  (AI probability: 98.6%)

--- Business ---
AI-generated  (AI probability: 96.6%)

--- Conversational ---
AI-generated  (AI probability: 96.9%)
